In [ ]:
import brightway2 as bw
from premise import *
import os
import wurst
import time
import copy

In [ ]:
startTime = time.time() # just to see how much time the code takes to run (this is the start)

In [ ]:
bw.projects.set_current('Prospective LCA v2') # set current project

In [ ]:
bioSphereDBName = 'biosphere3'
bioSphereDB = bw.Database(bioSphereDBName) # import the biosphere database
ecoInventBase2020DBName = 'ecoinvent 3.8 cutoff image SSP2-Base 2020'
ecoInventBase2020DB = bw.Database(ecoInventBase2020DBName) # import the ecoinvent database (image base 2020)

In [ ]:
# import base inventories from excel
excelImport = bw.ExcelImporter(os.path.join('..', 'Raw data', 'lci-Abhi image SSP2-Base 2020 new.xlsx'))
excelImport.apply_strategies()
excelImport.match_database(ecoInventBase2020DBName, fields = ('name', 'reference product', 'unit', 'location')) # match flows from ecoinvent database
excelImport.match_database(bioSphereDBName, fields = ('name', 'unit', 'categories')) # match flows from biosphere database
excelImport.match_database(fields = ('name', 'reference product', 'unit', 'location')) # match flows from my database
excelImport.statistics()

In [ ]:
# modify inventories to substitute 'reference product' with 'product' in all the exchanges
for ds in excelImport:
    for exchange in ds['exchanges']:
        if 'reference product' in exchange:
            exchange['product'] = exchange.pop('reference product')

In [ ]:
lciBase2020DBName = 'lci-Abhi image SSP2-Base 2020'

In [ ]:
globalDB = excelImport.data # add all inventories
ecoInventBase2020DBDict = [ds.as_dict() for ds in ecoInventBase2020DB] # convert ecoinvent database to dictionary
bioSphereDBDict = [ds.as_dict() for ds in bioSphereDB] # convert biosphere database to dictionary # maybe not needed?

newLocations = {'CN' : 'China',
                'IN' : 'India',
                'RER' : 'Europe',
                'US' : 'United States',
                'CH' : 'Switzerland',
                'DE' : 'Germany',
                'GB' : 'United Kingdom',
                'ZA' : 'South Africa',
                'AU' : 'Australia',
                'RU' : 'Russia'}

remindLocations = {'CHA' : 'China',
                   'IND' : 'India',
                   'EUR' : 'Europe',
                   'USA' : 'United States',
                   'NEU' : 'Switzerland',
                   'EUR' : 'Germany',
                   'EUR' : 'United Kingdom',
                   'SSA' : 'South Africa',
                   'CAZ' : 'Australia',
                   'REF' : 'Russia'}

imageLocations = {'CHN' : 'China',
                  'INDIA' : 'India',
                  'WEU' : 'Europe',
                  'USA' : 'United States',
                  'WEU' : 'Switzerland',
                  'WEU' : 'Germany',
                  'WEU' : 'United Kingdom',
                  'SAF' : 'South Africa',
                  'OCE' : 'Australia',
                  'RUS' : 'Russia'}

DSToRegionalize = globalDB

regionalizedDS = []
for ds in DSToRegionalize:
    for loc in newLocations:
        dsCopy = wurst.transformations.geo.copy_to_new_location(ds, loc)
        regionalizedDS.append(dsCopy)        

# Relink technosphere exchanges to the new locations
for ds in regionalizedDS:
    for exchange in ds['exchanges']:
        if exchange['type'] == 'technosphere':
            if exchange['database'] == ecoInventBase2020DBName:
                dsOutput = [dsInstance for dsInstance in ecoInventBase2020DBDict if exchange['name'] == dsInstance['name'] 
                                and exchange['product'] == dsInstance['reference product'] 
                                and ds['location'] == dsInstance['location']]
                if not dsOutput and 'market group for electricity' in exchange['name']:
                    exchange['name'] = exchange['name'].replace('group ', '')
                    dsOutput = [dsInstance for dsInstance in ecoInventBase2020DBDict if exchange['name'] == dsInstance['name'] 
                                and exchange['product'] == dsInstance['reference product'] 
                                and ds['location'] == dsInstance['location']]
            elif exchange['database'] == lciBase2020DBName:
                dsOutput = [dsInstance for dsInstance in regionalizedDS if exchange['name'] == dsInstance['name']
                                and ds['location'] == dsInstance['location']]
            if dsOutput:
                    exchange.update({'location' : dsOutput[0]['location']})

In [ ]:
db = globalDB + regionalizedDS
DBLinked = copy.deepcopy(db)

production = lambda x : x['type'] == 'production'
technosphere = lambda x : x['type'] == 'technosphere'
biosphere = lambda x : x['type'] == 'biosphere'

# linking exchanges to database inventories
for ds in DBLinked:
    for exchange in filter(production, ds['exchanges']):
        exchange.update({'input' : (ds['database'], ds['code'])})
    for exchange in filter(technosphere, ds['exchanges']):
        try:
            exchangeLink = wurst.get_one(db + ecoInventBase2020DBDict,
                                         wurst.equals('name', exchange['name']),
                                         wurst.equals('reference product', exchange['product']),
                                         wurst.equals('location', exchange['location']))
            exchange.update({'input' : (exchangeLink['database'], exchangeLink['code'])})
        except Exception:
            print(exchange['name'], exchange['product'], exchange['location'])
            raise
    # biosphere links maybe not needed
    for exchange in filter(biosphere, ds['exchanges']):
        if 'input' not in exchange:
            try:
                exchangeLink = [ds['code'] for ds in bioSphereDBDict if ds['name'] == exchange['name'] and
                                                                        ds['unit'] == exchange['unit'] and
                                                                        ds['categories'] == exchange['categories']][0]
                exchange.update({'input' : (bioSphereDBName, exchangeLink)})
            except Exception:
                print(exchange['name'], exchange['unit'], exchange['categories'])
                raise

In [ ]:
if lciBase2020DBName in bw.databases:
    del bw.databases[lciBase2020DBName]  
wurst.write_brightway2_database(DBLinked, lciBase2020DBName) # write database

In [ ]:
lciBase2020DB = wurst.extract_brightway2_databases(lciBase2020DBName)

In [ ]:
databaseNames = bw.databases
ecoInventDBNames = []
premiseDBNames = []
for databaseName in databaseNames:
    if 'Abhi' not in databaseName and 'biosphere3' not in databaseName and ('image' in databaseName or 'remind' in databaseName):
        ecoInventDBNames.append(databaseName)
        premiseDBNames.append(databaseName.replace('ecoinvent 3.8 cutoff ', ''))
ecoInventDBNames.sort()
premiseDBNames.sort()

In [ ]:
for premiseDBName in premiseDBNames:
    print('linking database ' + premiseDBName + '...')
    premiseDBDict = [ds.as_dict() for ds in bw.Database('ecoinvent 3.8 cutoff ' + premiseDBName)]

    DBLinked = copy.deepcopy(lciBase2020DB)

    production = lambda x : x['type'] == 'production'  
    technosphere = lambda x : x['type'] == 'technosphere'
    biosphere = lambda x : x['type'] == 'biosphere'

    for ds in DBLinked:
        for exchange in filter(technosphere, ds['exchanges']):
            try:
                exchangeLink = wurst.get_one(lciBase2020DB + premiseDBDict,
                                            wurst.equals('name', exchange['name']),
                                            wurst.equals('reference product', exchange['product']),
                                            wurst.equals('location', exchange['location'])
                                            )
                exchange.update({'input' : (exchangeLink['database'], exchangeLink['code'])})
                exchange.update({'database' : exchangeLink['database']})
            except Exception:
                print(exchange['name'], exchange['reference product'], exchange['location'])
                raise
        for exchange in filter(biosphere, ds['exchanges']):
            if 'input' not in exchange:
                try:
                    exchangeLink = [ds['code'] for ef in bioSphereDBDict if ds['name'] == exchange['name'] and 
                                                                            ds['unit'] == exchange['unit'] and 
                                                                            ds['categories'] == exchange['categories']][0]
                    exchange.update({'input': ('biosphere3', exchangeLink)})   
                except Exception:
                    print(exchange['name'], exchange['unit'], exchange['categories'])
                    raise
    dbName = 'lci-Abhi ' + premiseDBName
    if dbName in bw.databases:
        del bw.databases[dbName]
    wurst.write_brightway2_database(DBLinked, dbName)
    print(premiseDBName + ' linking and writing successful!')


In [ ]:
endTime = time.time() # end time
elapsedTime = endTime - startTime # calculate elapsed time
print(f'Elapsed time: {elapsedTime/3600:.2f} hours') 